# Preparación de Datos para Modelamiento
### Estadística Computacional para la Toma de Decisiones

**Magíster en Ciencia de Datos e Inteligencia Artificial**
Universidad Andrés Bello

*Maidana, J.P. (2026). Apunte — versión Jupyter Notebook.*

---

Este notebook es la versión **ejecutable** del apunte. En lugar de mostrar fragmentos sueltos que referencian un `datos.csv` inexistente, construimos **un único dataset "sucio" de clientes** y lo seguimos de principio a fin por todo el flujo de preparación: inspección, limpieza, tratamiento de faltantes, codificación, escalamiento, división train/test y un **pipeline completo** que termina entrenando un modelo. Todo corre de verdad.

> 💡 Busca los comentarios **`👉 EXPERIMENTA`**: marcan parámetros que vale la pena cambiar para ver cómo reacciona la preparación.

**Requisitos:** sólo `numpy`, `pandas`, `matplotlib` y `scikit-learn` — todo viene preinstalado en Google Colab, así que corre sin instalar nada.

## Configuración del entorno

Importamos librerías, fijamos una semilla y definimos un estilo de gráficos. **Ejecuta esta celda primero.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titleweight"] = "bold"

import sklearn
print("Entorno listo.  numpy", np.__version__, "| pandas", pd.__version__,
      "| sklearn", sklearn.__version__)

# 1. Introducción: El 80% del Trabajo

Existe un dicho popular en ciencia de datos:

> *"El 80% del tiempo se gasta en preparar los datos, solo el 20% en modelar y analizar."*

Esta no es una exageración. La **preparación de datos** es la fase más crítica y laboriosa de cualquier proyecto de machine learning. Un modelo sofisticado con datos mal preparados producirá resultados pobres, mientras que un modelo simple con datos bien preparados puede ser sorprendentemente efectivo.

> ### 📘 Definición — Preparación de Datos
> Es el proceso de transformar datos **crudos** (tal como se recolectan) en datos **listos para modelar** (en el formato que los algoritmos requieren). Incluye: limpieza de datos, tratamiento de valores faltantes, codificación de variables categóricas, escalamiento de variables numéricas, división train/test y creación de nuevas variables (*feature engineering*).

## 1.1 Flujo de Preparación

El recorrido conceptual va de datos crudos a datos listos para modelar:

In [ ]:
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(13, 3.2))
ax.set_xlim(0, 13); ax.set_ylim(0, 3); ax.axis("off")

etapas = [
    (1.4,  "Datos\nCrudos",      "Errores\nFaltantes\nCategoricas", "#f4a6a6"),
    (4.7,  "Datos\nLimpios",     "Sin errores\nCompletos",          "#f7e1a0"),
    (8.0,  "Datos\nTransform.",  "Codificadas\nEscaladas",          "#a9c7f0"),
    (11.3, "Datos\nListos",      "Train\nTest",                     "#a9e0a9"),
]
for x, titulo, detalle, color in etapas:
    ax.add_patch(FancyBboxPatch((x - 1.1, 0.7), 2.2, 1.6,
                 boxstyle="round,pad=0.05", fc=color, ec="black", lw=1.5))
    ax.text(x, 1.9, titulo, ha="center", va="center", fontweight="bold", fontsize=10)
    ax.text(x, 1.15, detalle, ha="center", va="center", fontsize=8)

for x0, x1, etiqueta in [(2.5, 3.6, "Limpiar"), (5.8, 6.9, "Transformar"), (9.1, 10.2, "Dividir")]:
    ax.annotate("", xy=(x1, 1.5), xytext=(x0, 1.5),
                arrowprops=dict(arrowstyle="-|>", lw=2, color="black"))
    ax.text((x0 + x1) / 2, 1.72, etiqueta, ha="center", fontsize=8.5, style="italic")

ax.set_title("Flujo de preparacion de datos", fontweight="bold")
plt.show()

> ### 📝 Lo que aprenderás en este notebook
> Identificar y corregir problemas comunes en datos; estrategias para tratar valores faltantes; codificar categóricas (one-hot, label encoding); escalar numéricas (estandarización, normalización); dividir correctamente en train/test con sklearn; buenas prácticas y errores comunes; y un **pipeline completo reproducible**.

# 2. Limpieza de Datos

La limpieza es el primer paso crítico: los datos crudos casi siempre tienen problemas que deben corregirse.

## 2.1 Problemas Comunes

| Problema | Descripción | Ejemplo |
|---|---|---|
| Valores faltantes | Celdas vacías o NaN | edad = NaN |
| Duplicados | Filas repetidas | Mismo cliente 2 veces |
| Errores de tipeo | Inconsistencias en texto | "Hombre", "hombre", "H" |
| Valores imposibles | Fuera de rango lógico | edad = −5, edad = 250 |
| Tipos incorrectos | Numérica como texto | precio = "\$1,234.56" |
| Outliers extremos | Valores atípicos | ingreso = \$10,000,000,000 |
| Inconsistencias | Contradicciones | edad = 10, ocupación = "CEO" |

Para practicar con datos realistas, **fabricamos un dataset de clientes que contiene todos estos problemas a la vez**. Este dataset será el hilo conductor de todo el notebook.

In [ ]:
# ====================================================================
#  Dataset "sucio" de clientes (hilo conductor de todo el notebook)
# ====================================================================
np.random.seed(42)
n = 200

# 1) Valores "latentes" LIMPIOS (sirven para generar el target sin contaminarlo)
edad_real    = np.random.randint(18, 75, n).astype(float)
ingreso_real = np.random.normal(55000, 18000, n).round(2)          # USD anuales
educacion    = np.random.choice(["Primaria", "Secundaria", "Universitaria"],
                                n, p=[0.2, 0.5, 0.3])
ciudad       = np.random.choice(["Santiago", "Valparaiso", "Concepcion"], n)

# Target: probabilidad de compra ligada a ingreso y educacion (clase positiva ~20%)
nivel = {"Primaria": 0, "Secundaria": 1, "Universitaria": 2}
z = -4 + 0.00004 * ingreso_real + 0.5 * np.array([nivel[e] for e in educacion])
compra = (np.random.random(n) < 1 / (1 + np.exp(-z))).astype(int)

# Genero "sucio": mismas personas, etiquetas inconsistentes
genero = np.random.choice(["Hombre", "hombre", "H", "Mujer", "mujer", "F"], n)

df_raw = pd.DataFrame({
    "id_cliente": np.arange(1, n + 1),
    "edad": edad_real, "ingreso": ingreso_real,
    "genero": genero, "ciudad": ciudad, "educacion": educacion,
    "compra": compra,
})

# 2) Inyectamos problemas tipicos de datos crudos -----------------------------
rng = np.random.default_rng(0)
df_raw.loc[rng.choice(n, 15, replace=False), "edad"]    = np.nan   # faltantes
df_raw.loc[rng.choice(n, 20, replace=False), "ingreso"] = np.nan
df_raw.loc[rng.choice(n, 10, replace=False), "genero"]  = np.nan
df_raw.loc[rng.choice(n,  8, replace=False), "ciudad"]  = np.nan
df_raw.loc[3, "edad"] = -5                      # edad imposible (negativa)
df_raw.loc[7, "edad"] = 250                     # edad imposible (altisima)
df_raw.loc[11, "ingreso"] = -999                # valor centinela
df_raw.loc[15, "ingreso"] = 10_000_000_000      # outlier absurdo

# Duplicados: repetimos 5 clientes
df_raw = pd.concat([df_raw, df_raw.sample(5, random_state=1)], ignore_index=True)

print(f"Dataset crudo: {df_raw.shape[0]} filas, {df_raw.shape[1]} columnas")
df_raw.head()

## 2.2 Exploración Inicial

> ### 🔧 Método — Inspección con Pandas
> Lo primero en cualquier proyecto: mirar la forma, los tipos, las estadísticas, los faltantes y los duplicados.

In [ ]:
print("=" * 60)
print("INFORMACION GENERAL")
print("=" * 60)
print(f"Forma: {df_raw.shape}")
print("\nTipos de datos:")
print(df_raw.dtypes)

print("\nEstadisticas descriptivas (variables numericas):")
print(df_raw.describe())

print("\nVALORES FALTANTES por columna:")
print(df_raw.isnull().sum())
total_na = df_raw.isnull().sum().sum()
print(f"\nTotal de faltantes: {total_na}  ({total_na / df_raw.size * 100:.2f}% del dataset)")
print(f"Filas duplicadas: {df_raw.duplicated().sum()}")

## 2.3 Detección de Valores Imposibles

Fíjate en cómo `describe()` ya delató el problema: la edad máxima es 250 y el ingreso máximo es de miles de millones. Los detectamos explícitamente.

In [ ]:
print("DETECCION DE VALORES IMPOSIBLES\n")

# Edad
print(f"Edad    -> min: {df_raw['edad'].min()}, max: {df_raw['edad'].max()}")
print(f"  {(df_raw['edad'] < 0).sum()} edad(es) negativa(s)")
print(f"  {(df_raw['edad'] > 120).sum()} edad(es) > 120 anios")

# Ingreso
print(f"\nIngreso -> min: ${df_raw['ingreso'].min():,.2f}, max: ${df_raw['ingreso'].max():,.2f}")
print(f"  {(df_raw['ingreso'] < 0).sum()} ingreso(s) negativo(s) (posible centinela -999)")
print(f"  {(df_raw['ingreso'] > 10_000_000).sum()} ingreso(s) > $10M (outlier sospechoso)")

## 2.4 Corrección de Problemas

Aplicamos las correcciones **estructurales**: eliminar duplicados, marcar valores imposibles como `NaN` (los imputaremos *después* de dividir train/test, para no filtrar información) y estandarizar el texto inconsistente de `genero`.

In [ ]:
df_clean = df_raw.copy()
filas_inicio = len(df_clean)

# 1) Eliminar duplicados (exactos y por id_cliente)
df_clean = df_clean.drop_duplicates()
df_clean = df_clean.drop_duplicates(subset=["id_cliente"], keep="first")

# 2) Marcar valores imposibles como NaN (se imputan luego, sin data leakage)
df_clean.loc[(df_clean["edad"] < 0) | (df_clean["edad"] > 120), "edad"] = np.nan
df_clean.loc[df_clean["ingreso"] <= 0, "ingreso"] = np.nan
df_clean.loc[df_clean["ingreso"] > 10_000_000, "ingreso"] = np.nan

# 3) Estandarizar 'genero' (texto inconsistente -> 2 categorias limpias)
df_clean["genero"] = df_clean["genero"].str.lower().str.strip()
mapeo_genero = {"h": "masculino", "hombre": "masculino", "m": "masculino",
                "mujer": "femenino", "f": "femenino"}
df_clean["genero"] = df_clean["genero"].map(mapeo_genero)

print(f"Filas: {filas_inicio} -> {len(df_clean)} "
      f"(se eliminaron {filas_inicio - len(df_clean)} duplicadas)")
print("\nCategorias de 'genero' tras estandarizar:")
print(df_clean["genero"].value_counts(dropna=False))
print("\nFaltantes restantes (se imputaran DESPUES de dividir train/test):")
print(df_clean[["edad", "ingreso", "genero", "ciudad"]].isnull().sum())

In [ ]:
# Mini-demostracion del problema "tipo incorrecto": numerica guardada como texto
serie_texto = pd.Series(["$1,234.56", "$987.00", "$2,500.10"])
serie_num = serie_texto.str.replace(r"[$,]", "", regex=True).astype(float)
print("Texto   :", serie_texto.tolist())
print("Numerico:", serie_num.tolist(), "  dtype:", serie_num.dtype)

# 3. Tratamiento de Valores Faltantes

Los valores faltantes son uno de los problemas más comunes. Debemos decidir qué hacer con ellos.

## 3.1 Tipos de Missingness

> ### 📘 Definición — Tipos de Missingness
> **MCAR (Missing Completely At Random):** faltan al azar, sin patrón (un sensor falla aleatoriamente). Impacto menor.
> **MAR (Missing At Random):** faltan según *otras* variables observadas (personas mayores no responden el ingreso). Impacto moderado, puede sesgar.
> **MNAR (Missing Not At Random):** faltan según el *propio* valor de la variable (ingresos muy altos no se reportan). Impacto alto, sesgo severo.

## 3.2 Estrategias de Tratamiento

| Estrategia | Cuándo usar | Ventaja / Desventaja |
|---|---|---|
| Eliminar filas | Pocos faltantes (<5%), MCAR | + Simple / − Pierde datos |
| Eliminar columnas | Muchos faltantes (>50%) | + Limpia / − Pierde info |
| Imputar con constante | Categórica | + Simple / − Crea categoría |
| Imputar media/mediana | Numérica simétrica | + Preserva promedio / − Reduce varianza |
| Imputar con moda | Categórica | + Mantiene distribución / − Sesgo a moda |
| Imputación avanzada | Datos importantes, MAR | + Preserva relaciones / − Complejo |

## 3.3 Implementación

Veamos la mecánica de cada estrategia en un ejemplo mínimo y legible:

In [ ]:
from sklearn.impute import SimpleImputer

# Ejemplo minimo para ver la mecanica de cada estrategia
ej = pd.DataFrame({
    "edad":    [25, 30, np.nan, 45, 50, np.nan, 35],
    "ingreso": [30000, 45000, 50000, np.nan, 60000, 55000, np.nan],
    "genero":  ["M", "F", "M", np.nan, "F", "M", "F"],
    "ciudad":  ["Santiago", np.nan, "Santiago", "Valparaiso", np.nan, "Santiago", "Concepcion"],
})
print("Faltantes por columna:")
print(ej.isnull().sum().to_string())
print(f"\nTras dropna(): {ej.dropna().shape[0]} filas (de {len(ej)})")

# Imputacion con asignacion directa (sin inplace ni chained assignment)
ej_imp = ej.copy()
ej_imp["edad"]    = ej_imp["edad"].fillna(ej_imp["edad"].mean())        # media
ej_imp["ingreso"] = ej_imp["ingreso"].fillna(ej_imp["ingreso"].median())  # mediana
ej_imp["genero"]  = ej_imp["genero"].fillna(ej_imp["genero"].mode()[0])   # moda
ej_imp["ciudad"]  = ej_imp["ciudad"].fillna("Desconocido")                # constante
print("\nTras imputar:")
print(ej_imp)

# Lo mismo con SimpleImputer de sklearn (lo que integraremos en el pipeline final)
imp_num = SimpleImputer(strategy="mean")
num_imp = pd.DataFrame(imp_num.fit_transform(ej[["edad", "ingreso"]]),
                       columns=["edad", "ingreso"])
print("\nImputacion numerica con SimpleImputer:")
print(num_imp)

> ### ⚠️ Cuidado con el *Data Leakage*
> Al imputar **nunca** calcules estadísticas (media, mediana) usando *todo* el dataset. El orden correcto es: (1) dividir en train/test, (2) calcular estadísticas **sólo en train**, (3) aplicar esas estadísticas a test. Si calculas la media con todo el dataset, información del test "se filtra" al entrenamiento.
>
> ```python
> # CORRECTO
> X_train, X_test, y_train, y_test = train_test_split(X, y)
> imputer = SimpleImputer(strategy='mean')
> imputer.fit(X_train)            # ajusta SOLO en train
> X_train = imputer.transform(X_train)
> X_test  = imputer.transform(X_test)
>
> # INCORRECTO: imputa con todo el dataset ANTES de dividir
> X = imputer.fit_transform(X)
> X_train, X_test, ... = train_test_split(X, y)
> ```

Demostrémoslo de verdad sobre nuestro dataset de clientes:

In [ ]:
from sklearn.model_selection import train_test_split

# Forma CORRECTA: dividir primero, ajustar el imputador SOLO con train
X = df_clean[["edad", "ingreso"]]
y = df_clean["compra"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                          random_state=42, stratify=y)

imp = SimpleImputer(strategy="mean")
imp.fit(X_tr)                     # <- estadisticas SOLO de train
X_tr_imp = imp.transform(X_tr)
X_te_imp = imp.transform(X_te)    # <- test usa las medias de train

print("Medias aprendidas (solo de train):")
for col, m in zip(X.columns, imp.statistics_):
    print(f"  {col:8s}: {m:,.2f}")

print(f"\nMedia de 'edad' segun train      : {X_tr['edad'].mean():.2f}")
print(f"Media de 'edad' segun TODO el set: {X['edad'].mean():.2f}")
print("Usar la media de 'todo' para imputar test seria DATA LEAKAGE.")

# 4. Codificación de Variables Categóricas

Los algoritmos trabajan con números, no con texto. Debemos convertir las variables categóricas a formato numérico.

> ### 📘 Definición — Nominales vs. Ordinales
> **Nominales** (sin orden): género, color, país → **One-Hot Encoding**.
> **Ordinales** (con orden natural): educación (primaria < secundaria < universitaria), satisfacción (bajo < medio < alto) → **Label / Ordinal Encoding**.

## 4.1 One-Hot Encoding

Convierte cada categoría en una columna binaria (0/1). Una variable `color` con valores {Rojo, Verde, Azul} se vuelve 3 columnas. **Ventaja:** no impone orden artificial. **Desventaja:** aumenta la dimensionalidad.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

ej = pd.DataFrame({
    "nombre": ["Ana", "Juan", "Maria", "Pedro", "Laura"],
    "genero": ["F", "M", "F", "M", "F"],
    "ciudad": ["Santiago", "Valparaiso", "Santiago", "Concepcion", "Santiago"],
    "edad":   [25, 30, 35, 28, 32],
})

# OPCION 1: pd.get_dummies (rapido y directo)
df_dummies = pd.get_dummies(ej, columns=["genero", "ciudad"], prefix=["genero", "ciudad"])
print("get_dummies:")
print(df_dummies)

# OPCION 2: OneHotEncoder de sklearn (mas control, integrable en pipelines)
encoder = OneHotEncoder(sparse_output=False, drop="first")
X_enc = encoder.fit_transform(ej[["genero", "ciudad"]])
nombres = encoder.get_feature_names_out(["genero", "ciudad"])
df_enc = pd.DataFrame(X_enc, columns=nombres)
print("\nOneHotEncoder (drop='first' elimina una columna para evitar redundancia):")
print(pd.concat([ej[["nombre", "edad"]], df_enc], axis=1))

## 4.2 Label / Ordinal Encoding

Convierte cada categoría en un entero. Úsalo **sólo para variables ordinales**. Ojo con la diferencia: `LabelEncoder` asigna enteros en orden **alfabético** (puede no respetar el orden real), mientras que `OrdinalEncoder` permite **definir el orden a mano**.

In [ ]:
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

ej_ord = pd.DataFrame({
    "educacion":    ["Secundaria", "Universitaria", "Primaria", "Universitaria", "Secundaria"],
    "satisfaccion": ["Baja", "Alta", "Media", "Alta", "Media"],
})

# OPCION 1: LabelEncoder (orden alfabetico -> NO necesariamente el orden logico)
le = LabelEncoder()
ej_ord["educacion_cod"] = le.fit_transform(ej_ord["educacion"])
print("LabelEncoder (alfabetico):")
print(ej_ord[["educacion", "educacion_cod"]])
print("Mapeo:", dict(zip(le.classes_, range(len(le.classes_)))))

# OPCION 2: OrdinalEncoder con el ORDEN correcto definido explicitamente
oe = OrdinalEncoder(categories=[["Primaria", "Secundaria", "Universitaria"],
                                ["Baja", "Media", "Alta"]])
cod = oe.fit_transform(ej_ord[["educacion", "satisfaccion"]])
df_cod = pd.DataFrame(cod, columns=["educacion_ord", "satisfaccion_ord"])
print("\nOrdinalEncoder (orden natural respetado):")
print(pd.concat([ej_ord[["educacion", "satisfaccion"]], df_cod], axis=1))

> ### ⭐ Regla de decisión
> El siguiente diagrama resume cómo elegir la codificación según el tipo de variable y su cardinalidad:

In [ ]:
from matplotlib.patches import FancyBboxPatch, Polygon

fig, ax = plt.subplots(figsize=(10, 7))
ax.set_xlim(0, 10); ax.set_ylim(0, 9); ax.axis("off")

def caja(x, y, t, color, w=2.7, h=0.95):
    ax.add_patch(FancyBboxPatch((x - w/2, y - h/2), w, h,
                 boxstyle="round,pad=0.05", fc=color, ec="black", lw=1.3))
    ax.text(x, y, t, ha="center", va="center", fontsize=9, fontweight="bold")

def rombo(x, y, t, w=3.2, h=1.4):
    ax.add_patch(Polygon([(x, y + h/2), (x + w/2, y), (x, y - h/2), (x - w/2, y)],
                 closed=True, fc="#f7e1a0", ec="black", lw=1.3))
    ax.text(x, y, t, ha="center", va="center", fontsize=8.5)

def flecha(p0, p1, etq="", dx=0.0):
    ax.annotate("", xy=p1, xytext=p0, arrowprops=dict(arrowstyle="-|>", lw=1.6, color="black"))
    if etq:
        ax.text((p0[0]+p1[0])/2 + dx, (p0[1]+p1[1])/2, etq, ha="center", fontsize=8.5,
                style="italic", bbox=dict(fc="white", ec="none", pad=0.4))

caja(5, 8.3, "Variable categorica", "#a9c7f0")
rombo(5, 6.4, "¿Es ordinal?")
caja(2.0, 4.3, "Label / Ordinal\nEncoding", "#a9e0a9")
rombo(7.3, 4.5, "¿Muchas categorias\n(> 10)?")
caja(5.8, 2.2, "One-Hot\nEncoding", "#a9e0a9")
caja(9.0, 2.2, "Target Encoding\no agrupacion", "#f6c98a")

flecha((5, 7.83), (5, 7.1))
flecha((3.9, 6.05), (2.5, 4.8), "Si", dx=-0.25)
flecha((6.1, 6.05), (7.1, 5.2), "No", dx=0.25)
flecha((6.7, 3.9), (6.0, 2.7), "No", dx=-0.25)
flecha((7.9, 3.9), (8.8, 2.7), "Si", dx=0.25)

ax.set_title("¿Como codificar una variable categorica?", fontweight="bold")
plt.show()

# 5. Escalamiento de Variables Numéricas

Algunas variables tienen escalas muy diferentes; el escalamiento las pone en rangos comparables.

> ### 📘 Definición — ¿Por qué escalar?
> **Edad** (rango ~60) vs. **Ingreso** (rango ~180,000): el ingreso domina por su magnitud aunque ambas importen igual. **Necesario** en: modelos por distancia (KNN, K-means), gradiente descendente (redes neuronales), regularización (Ridge, Lasso). **NO necesario** en: regresión lineal sin regularización, árboles de decisión.

## 5.1 Métodos de Escalamiento

| Método | Fórmula | Resultado |
|---|---|---|
| **StandardScaler** | $z = \dfrac{x - \mu}{\sigma}$ | Media 0, desv. 1. Conserva outliers. |
| **MinMaxScaler** | $x' = \dfrac{x - x_{min}}{x_{max} - x_{min}}$ | Rango [0, 1]. Sensible a outliers. |
| **RobustScaler** | $x' = \dfrac{x - Q_2}{Q_3 - Q_1}$ | Usa mediana y cuartiles. Robusto a outliers. |

Comparemos los tres sobre datos **con outliers** para ver sus diferencias:

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

np.random.seed(42)
n = 100
edad    = np.append(np.random.normal(40, 10, n), [5, 95, 100])          # + outliers
ingreso = np.append(np.random.normal(50000, 15000, n), [10000, 150000, 200000])
datos = pd.DataFrame({"edad": edad, "ingreso": ingreso})

for nombre, sc in [("StandardScaler", StandardScaler()),
                   ("MinMaxScaler",   MinMaxScaler()),
                   ("RobustScaler",   RobustScaler())]:
    res = pd.DataFrame(sc.fit_transform(datos), columns=datos.columns)
    print(f"{nombre}:")
    print(res.describe().loc[["mean", "std", "min", "max"]].round(2))
    print()

In [ ]:
# Como trata cada metodo los outliers (variable 'edad')
versiones = [
    ("Original",       datos["edad"].values),
    ("StandardScaler", StandardScaler().fit_transform(datos)[:, 0]),
    ("MinMaxScaler",   MinMaxScaler().fit_transform(datos)[:, 0]),
    ("RobustScaler",   RobustScaler().fit_transform(datos)[:, 0]),
]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
for ax, (titulo, valores) in zip(axes, versiones):
    ax.hist(valores, bins=25, color="#5b8def", edgecolor="white")
    ax.set_title(titulo, fontsize=11)
    ax.grid(alpha=0.3)
axes[0].set_ylabel("frecuencia")
fig.suptitle("Efecto de cada escalador sobre 'edad' (outliers en 5, 95, 100)",
             fontweight="bold")
plt.tight_layout()
plt.show()

# 👉 EXPERIMENTA: agrega un outlier extremo (p.ej. 500) y observa como MinMax
#    comprime casi todos los datos hacia 0, mientras RobustScaler apenas cambia.

> ### ⭐ ¿Cuál método elegir?
> **StandardScaler**: método por defecto, apropiado en la mayoría de casos. **MinMaxScaler**: cuando necesitas rango [0,1] específicamente. **RobustScaler**: cuando hay muchos outliers que quieres preservar pero no que dominen. **Para este curso:** usar **StandardScaler** como estándar, o no escalar si usas sólo regresión lineal sin regularización.

# 6. División Train/Test

La división correcta es **CRÍTICA**: es el mecanismo que garantiza una evaluación honesta.

> ### 📘 Definición — Train/Test Split
> Dividir en dos conjuntos **disjuntos**. **Train (70–80%)**: el modelo aprende de estos datos. **Test (20–30%)**: usado SÓLO para la evaluación final; el modelo nunca lo ve durante el entrenamiento, simulando datos futuros. Objetivo: estimar qué tan bien **generalizará** el modelo.

In [ ]:
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(11, 3.4))
ax.set_xlim(0, 11); ax.set_ylim(-2.6, 2); ax.axis("off")

ax.add_patch(FancyBboxPatch((0.3, 0.6), 10.4, 1.1, boxstyle="round,pad=0.02",
             fc="#a9c7f0", ec="#28527a", lw=1.5))
ax.text(5.5, 1.15, "Dataset completo (100%)", ha="center", va="center", fontweight="bold")

ax.add_patch(FancyBboxPatch((0.3, -1.3), 8.0, 1.1, boxstyle="round,pad=0.02",
             fc="#a9e0a9", ec="#2e7d32", lw=1.5))
ax.text(4.3, -0.75, "Train (70-80%)", ha="center", va="center", fontweight="bold")
ax.add_patch(FancyBboxPatch((8.6, -1.3), 2.1, 1.1, boxstyle="round,pad=0.02",
             fc="#f6c98a", ec="#b5651d", lw=1.5))
ax.text(9.65, -0.75, "Test (20-30%)", ha="center", va="center", fontweight="bold", fontsize=9)

ax.annotate("", xy=(4.3, -0.15), xytext=(3.0, 0.55), arrowprops=dict(arrowstyle="-|>", lw=1.8))
ax.annotate("", xy=(9.65, -0.15), xytext=(9.0, 0.55), arrowprops=dict(arrowstyle="-|>", lw=1.8))
ax.text(4.3, -1.8, "Entrenar el modelo", ha="center", fontsize=8.5, style="italic")
ax.text(9.65, -1.8, "Evaluar (datos 'nuevos')", ha="center", fontsize=8.5, style="italic")
ax.set_title("Division train / test", fontweight="bold")
plt.show()

## 6.1 Implementación con sklearn

Parámetros clave de `train_test_split`: `test_size` (proporción para test), `random_state` (reproducibilidad), `stratify` (mantener distribución de clases) y `shuffle` (mezclar antes de dividir, por defecto `True`).

In [ ]:
np.random.seed(42)
n = 100
ej = pd.DataFrame({
    "edad":    np.random.randint(20, 70, n),
    "ingreso": np.random.randint(20000, 100000, n),
    "genero":  np.random.choice(["M", "F"], n),
    "compra":  np.random.choice([0, 1], n, p=[0.7, 0.3]),
})
X = ej[["edad", "ingreso", "genero"]]
y = ej["compra"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}")

# Verificar que NO comparten filas (conjuntos disjuntos)
interseccion = set(X_train.index) & set(X_test.index)
print(f"Son disjuntos: {len(interseccion) == 0}")

## 6.2 División Estratificada

En clasificación con clases desbalanceadas queremos la **misma proporción de clases** en train y test. Comprobemos qué pasa con y sin `stratify`:

In [ ]:
prop_total = y.mean()
# Sin estratificar
_, _, ytr_a, yte_a = train_test_split(X, y, test_size=0.2, random_state=2)
# Con estratificar
_, _, ytr_b, yte_b = train_test_split(X, y, test_size=0.2, random_state=2, stratify=y)

print(f"Proporcion de compra=1 en el total: {prop_total:.3f}\n")
print("                      train     test")
print(f"Sin estratificar:     {ytr_a.mean():.3f}     {yte_a.mean():.3f}")
print(f"Con estratificar:     {ytr_b.mean():.3f}     {yte_b.mean():.3f}")
print("\nstratify=y acerca la proporcion de clases de test a la del total.")

> ### ⚠️ Errores comunes
> **(1) Transformar antes de dividir** — escalar/imputar con todo el dataset filtra información del test. **(2) Usar test para decisiones de modelado** — ajustar hiperparámetros mirando el test lo contamina. *Solución:* usa validación cruzada o un conjunto de validación separado para todas las decisiones.
>
> ```python
> # MAL: escala ANTES de dividir
> X_scaled = scaler.fit_transform(X)
> X_train, X_test = train_test_split(X_scaled)
> ```

# 7. Pipeline Completo de Preparación

Juntemos todo en un flujo reproducible y profesional, **sobre nuestro dataset de clientes ya limpio** (que aún conserva los `NaN` que decidimos imputar después del split). Un `ColumnTransformer` aplica un sub-pipeline distinto a las columnas numéricas y a las categóricas, y todo se ajusta **sólo en train**.

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Partimos del dataset YA limpio estructuralmente (Seccion 2); aun tiene NaN
X = df_clean.drop(columns=["id_cliente", "compra"])
y = df_clean["compra"]

# Identificar tipos de columna automaticamente
num_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X.select_dtypes(include=["object", "str", "category"]).columns.tolist()
print("Numericas  :", num_features)
print("Categoricas:", cat_features)

# DIVIDIR ANTES DE TRANSFORMAR
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Sub-pipelines por tipo de variable
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler",  StandardScaler()),
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(drop="first", handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("num", num_pipe, num_features),
    ("cat", cat_pipe, cat_features),
])

# Ajustar SOLO en train, transformar ambos
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep  = preprocessor.transform(X_test)
print(f"\nX_train preparado: {X_train_prep.shape}")
print(f"X_test preparado : {X_test_prep.shape}")
print("Datos listos para modelar: sin NaN, categoricas codificadas y numericas escaladas.")

In [ ]:
from sklearn.linear_model import LogisticRegression

# Pipeline COMPLETO: preprocesamiento + modelo en UN solo objeto reproducible
modelo = Pipeline([
    ("prep", preprocessor),
    ("clf",  LogisticRegression(max_iter=1000)),
])
modelo.fit(X_train, y_train)   # imputa, codifica, escala y entrena de una vez

print(f"Accuracy en train: {modelo.score(X_train, y_train):.3f}")
print(f"Accuracy en test : {modelo.score(X_test, y_test):.3f}")
print("\nUn unico objeto 'modelo' encapsula TODO el preprocesamiento + el clasificador.")
print("Ideal para produccion: mismas transformaciones garantizadas y sin data leakage.")

> ### ⭐ Ventajas del Pipeline
> **Reproducibilidad** (mismas transformaciones siempre), **sin data leakage** (ajusta sólo en train), **fácil de mantener** (todo el preprocesamiento en un objeto), **portable** (se guarda junto al modelo para producción) y **previene errores** (orden correcto garantizado).

# 8. Resumen y Mejores Prácticas

## 8.1 Checklist de Preparación

| Paso | Tarea | Acciones |
|---|---|---|
| 1 | Exploración | `info`, `describe`, `dtypes`, `head` |
| 2 | Limpieza | Eliminar duplicados, corregir tipos, filtrar imposibles |
| 3 | Faltantes | Identificar, decidir estrategia, imputar |
| 4 | Separar X, y | Definir target y features |
| 5 | **Train/Test** | **DIVIDIR ANTES DE TRANSFORMAR** |
| 6 | Codificar | One-hot para nominales, label para ordinales |
| 7 | Escalar | StandardScaler para numéricas (si es necesario) |
| 8 | Verificar | Shapes correctos, sin leakage, reproducible |

## 8.2 Reglas de Oro

1. **Train/Test primero**: dividir ANTES de cualquier transformación.
2. **Fit sólo en Train**: nunca ajustar transformadores con datos de test.
3. **Transform en ambos**: aplicar las mismas transformaciones a train y test.
4. **Stratify en clasificación**: mantener proporciones de clases.
5. **Imputar con sentido**: elegir estrategia según el tipo de variable.
6. **One-hot para nominales**, label sólo para ordinales.
7. **Escalar si es necesario**: no siempre se requiere.
8. **Pipeline siempre**: reproducibilidad y prevención de errores.
9. **Documentar decisiones**: por qué elegiste cada estrategia.
10. **Validar resultados**: verificar shapes, distribuciones, sin leakage.

## 8.3 Los 10 Errores Más Comunes

> ### ⚠️ Errores a evitar
> 1. Transformar antes de dividir train/test. 2. Calcular estadísticas en todo el dataset. 3. No usar `stratify` en clasificación desbalanceada. 4. Eliminar todas las filas con cualquier faltante. 5. Usar label encoding para variables nominales. 6. No verificar data leakage. 7. Olvidar que test simula datos futuros. 8. Escalar cuando no es necesario. 9. No guardar los transformadores para producción. 10. No validar que train y test son disjuntos.

> ### 💡 Cierre
> La preparación de datos es la base sobre la cual construimos modelos exitosos. Aunque consume la mayor parte del tiempo en un proyecto real, es una inversión que vale la pena: datos bien preparados garantizan modelos confiables y resultados válidos. La división correcta en train/test, el tratamiento apropiado de faltantes, la codificación cuidadosa de categóricas y el escalamiento cuando es necesario son habilidades fundamentales. Con estas herramientas y el uso de pipelines reproducibles, estás preparado para abordar cualquier proyecto de modelamiento con confianza y profesionalismo.

---
## Referencias bibliográficas

- Géron, A. (2022). *Hands-on machine learning with Scikit-Learn, Keras, and TensorFlow* (3rd ed.). O'Reilly Media.
- Kuhn, M., & Johnson, K. (2019). *Feature engineering and selection: A practical approach for predictive models*. CRC Press.
- McKinney, W. (2022). *Python for data analysis* (3rd ed.). O'Reilly Media.
- Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, 12, 2825–2830.
- Van Buuren, S. (2018). *Flexible imputation of missing data* (2nd ed.). CRC Press.